# Od chaosu plików do czystej ramki danych

## Obróbka danych — `pathlib`, `glob`, `re`, `pandas.str`

**Cel zajęć:** mamy katalog z pomiarami środowiskowymi z kilku stacji w Łodzi. Pliki są w różnych podkatalogach, mają różne formaty nazw, w środku różne separatory. Naszym zadaniem jest **wczytać wszystko, wyciągnąć metadane z nazw plików** (stacja, data, typ pomiaru) i **połączyć w jedną ramkę** gotową do analizy.

In [1]:
import zipfile
from pathlib import Path

zip_path = Path("dane_lodz.zip")
output_dir = Path("dane_lodz")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

## 1. Rozpoznanie terenu

Zacznijmy od obejrzenia, z czym mamy do czynienia.

In [2]:
from pathlib import Path

ROOT = Path("dane_lodz/dane_lodz")

# Sprawdzenie, czy katalog istnieje
if ROOT.exists() and ROOT.is_dir():
    # Lista podkatalogów
    for p in ROOT.iterdir():
        print(p)
else:
    print(f"Katalog '{ROOT}' nie istnieje.")

dane_lodz\dane_lodz\jakosc_powietrza
dane_lodz\dane_lodz\notatki.txt
dane_lodz\dane_lodz\temperatura
dane_lodz\dane_lodz\wilgotnosc


In [3]:
# Pokaż drzewo - kilka pierwszych plików z każdego katalogu
for subdir in sorted(ROOT.iterdir()):
    if subdir.is_dir():
        print(f"\n📁 {subdir.name}/")
        for f in sorted(subdir.iterdir())[:3]:
            print(f"   {f.name}")
        print("   ...")


📁 jakosc_powietrza/
   info.txt
   pm25-ST01_polesie-20250301.csv
   pm25-ST01_polesie-20250302.csv
   ...

📁 temperatura/
   _old_backup.csv.bak
   README.md
   temp_ST01_polesie_2025-03-01.csv
   ...

📁 wilgotnosc/
   humidity_ST01_polesie_v1_2025_03_01.txt
   humidity_ST01_polesie_v1_2025_03_02.txt
   humidity_ST01_polesie_v1_2025_03_04.txt
   ...


## 2. `pathlib` — podstawy

`pathlib` zastępuje `os.path`. Główna klasa: `Path`. Działa **cross-platform** — używaj `/` zamiast `os.path.join`.

In [4]:
# Operator / buduje ścieżki
plik = ROOT / "temperatura" / "temp_ST01_polesie_2025-03-01.csv"
print("Pełna ścieżka:    ", plik)
print("Nazwa pliku:      ", plik.name)
print("Sama nazwa:       ", plik.stem)
print("Rozszerzenie:     ", plik.suffix)
print("Katalog nadrzędny:", plik.parent)
print("Czy istnieje:     ", plik.exists())

Pełna ścieżka:     dane_lodz\dane_lodz\temperatura\temp_ST01_polesie_2025-03-01.csv
Nazwa pliku:       temp_ST01_polesie_2025-03-01.csv
Sama nazwa:        temp_ST01_polesie_2025-03-01
Rozszerzenie:      .csv
Katalog nadrzędny: dane_lodz\dane_lodz\temperatura
Czy istnieje:      True


### **Ćwiczenie 1**
Dla pliku `plik` powyżej wypisz:
- nazwę bez rozszerzenia
- nazwę katalogu nadrzędnego (samą nazwę, nie pełną ścieżkę)
- rozmiar pliku w bajtach (podpowiedź: `.stat().st_size`)



In [5]:
# Ćwiczenie 1 — pathlib
print("Nazwa bez rozszerzenia:  ", plik.stem)
print("Katalog nadrzędny (nazwa):", plik.parent.name)
print("Rozmiar pliku (bajty):   ", plik.stat().st_size)

Nazwa bez rozszerzenia:   temp_ST01_polesie_2025-03-01
Katalog nadrzędny (nazwa): temperatura
Rozmiar pliku (bajty):    281


## 3. `glob` i `rglob` — znajdowanie plików

- `Path.glob(pattern)` — w danym katalogu (jeden poziom)
- `Path.rglob(pattern)` — **rekurencyjnie** (we wszystkich podkatalogach)
- wzorce: `*` (cokolwiek), `?` (jeden znak), `[abc]` (zbiór znaków)

In [6]:
# Wszystkie pliki CSV w całym drzewie
csv_files = list(ROOT.rglob("*.csv"))
print(f"Znaleziono plików CSV: {len(csv_files)}")
for f in csv_files[:5]:
    print(" ", f)

Znaleziono plików CSV: 80
  dane_lodz\dane_lodz\jakosc_powietrza\pm25-ST01_polesie-20250301.csv
  dane_lodz\dane_lodz\jakosc_powietrza\pm25-ST01_polesie-20250302.csv
  dane_lodz\dane_lodz\jakosc_powietrza\pm25-ST01_polesie-20250303.csv
  dane_lodz\dane_lodz\jakosc_powietrza\pm25-ST01_polesie-20250304.csv
  dane_lodz\dane_lodz\jakosc_powietrza\pm25-ST01_polesie-20250305.csv


In [7]:
# Tylko pliki z temperatury
temp_files = list((ROOT / "temperatura").glob("temp_*.csv"))
print(f"Plików temperatury: {len(temp_files)}")

Plików temperatury: 40


In [8]:
# UWAGA: glob łapie też śmieci - np. backupy. Trzeba filtrować.
podejrzane = list(ROOT.rglob("*backup*"))
print("Pliki które chcemy zignorować:")
for f in podejrzane:
    print(" ", f)

Pliki które chcemy zignorować:
  dane_lodz\dane_lodz\temperatura\_old_backup.csv.bak


### **Ćwiczenie 2**
Znajdź:
1. Wszystkie pliki `.txt` w drzewie
2. Wszystkie pliki PM2.5 (zaczynają się od `pm25-`)
3. Wszystkie pliki temperatury, **z wyjątkiem** wersji `_v2`

In [9]:
# Ćwiczenie 2 — glob / rglob
# 1. Wszystkie pliki .txt w drzewie
txt_files = list(ROOT.rglob("*.txt"))
print("Plików .txt:", len(txt_files))

# 2. Wszystkie pliki PM2.5 (zaczynają się od 'pm25-')
pm_files = list(ROOT.rglob("pm25-*"))
print("Plików PM2.5:", len(pm_files))

# 3. Wszystkie pliki temperatury z WYJĄTKIEM wersji _v2
temp_all = list((ROOT / "temperatura").glob("temp_*.csv"))
temp_bez_v2 = [f for f in temp_all if "_v2" not in f.name]
print(f"Pliki temperatury: {len(temp_all)}, bez _v2: {len(temp_bez_v2)}")

Plików .txt: 42
Plików PM2.5: 40
Pliki temperatury: 40, bez _v2: 34


## 4. Wyrażenia regularne — krótkie powtórzenie

Regex to **język opisywania wzorców tekstu**. Najważniejsze elementy:

| Wzorzec | Znaczenie |
|---------|-----------|
| `\d` | cyfra (0-9) |
| `\w` | znak "słowny" (litera, cyfra, `_`) |
| `\s` | biały znak |
| `.` | dowolny znak |
| `+` | jeden lub więcej |
| `*` | zero lub więcej |
| `{n}` | dokładnie n |
| `{n,m}` | od n do m |
| `[abc]` | jeden ze znaków |
| `(...)` | grupa |
| `(?P<nazwa>...)` | **grupa nazwana** ← KLUCZOWE |

Funkcje z `re`:
- `re.search(pattern, text)` — szuka **gdziekolwiek** w tekście
- `re.match(pattern, text)` — szuka **od początku**
- `re.findall(pattern, text)` — wszystkie dopasowania
- `re.sub(pattern, replacement, text)` — zamiana

In [10]:
import re

# Prosty przykład: wyciągnij datę z nazwy pliku
nazwa = "temp_ST01_polesie_2025-03-01.csv"
wzorzec = r"(\d{4}-\d{2}-\d{2})"
m = re.search(wzorzec, nazwa)
print(m.group(1))

2025-03-01


**Grupy nazwane** to klucz do czytelnego kodu. Zamiast `m.group(1)`, `m.group(2)`... mamy `m.group('data')`.

In [11]:
# Wzorzec dla plików temperatury:
# temp_<STACJA>_<DATA>.csv  lub  temp_<STACJA>_<DATA>_v2.csv
wzorzec_temp = re.compile(
    r"temp_(?P<stacja>ST\d{2}_\w+?)_(?P<data>\d{4}-\d{2}-\d{2})(?:_v(?P<wersja>\d+))?\.csv"
)

for f in list((ROOT / "temperatura").glob("temp_*.csv"))[:5]:
    m = wzorzec_temp.match(f.name)
    if m:
        print(f.name)
        print("  →", m.groupdict())

temp_ST01_polesie_2025-03-01.csv
  → {'stacja': 'ST01_polesie', 'data': '2025-03-01', 'wersja': None}
temp_ST01_polesie_2025-03-02.csv
  → {'stacja': 'ST01_polesie', 'data': '2025-03-02', 'wersja': None}
temp_ST01_polesie_2025-03-03.csv
  → {'stacja': 'ST01_polesie', 'data': '2025-03-03', 'wersja': None}
temp_ST01_polesie_2025-03-04.csv
  → {'stacja': 'ST01_polesie', 'data': '2025-03-04', 'wersja': None}
temp_ST01_polesie_2025-03-05_v2.csv
  → {'stacja': 'ST01_polesie', 'data': '2025-03-05', 'wersja': '2'}


Rozbiór wzorca:
- `temp_` — literalny początek
- `(?P<stacja>ST\d{2}_\w+?)` — `ST` + 2 cyfry + `_` + nazwa (leniwa, żeby nie zjeść daty)
- `_` — separator
- `(?P<data>\d{4}-\d{2}-\d{2})` — data w formacie ISO
- `(?:_v(?P<wersja>\d+))?` — **opcjonalna** wersja, `(?:...)` to grupa nie-przechwytująca
- `\.csv` — kropka literalna (uciekamy `\.`) + rozszerzenie

### **ćwiczenie 3**

Napisz regex dla plików PM2.5 (przykład: `pm25-ST01_polesie-20250301.csv`).
Wyciągnij stację i datę. Uwaga: data nie ma separatorów!

I drugi — dla wilgotności (np. `humidity_ST02_widzew_v1_2025_03_05.txt`).
Tu wersja jest **w środku**, nie na końcu.

In [12]:
# Ćwiczenie 3 — regexy dla PM2.5 i wilgotności
import re

# PM2.5: pm25-ST01_polesie-20250301.csv  (data YYYYMMDD, BEZ separatorów)
wzorzec_pm = re.compile(r"pm25-(?P<stacja>ST\d{2}_\w+?)-(?P<data>\d{8})\.csv")
for przyklad in ["pm25-ST01_polesie-20250301.csv", "pm25-ST04_gorna-20250308.csv"]:
    print(przyklad, "->", wzorzec_pm.match(przyklad).groupdict())

# Wilgotność: humidity_ST02_widzew_v1_2025_03_05.txt  (wersja w ŚRODKU, data YYYY_MM_DD)
wzorzec_hum = re.compile(
    r"humidity_(?P<stacja>ST\d{2}_\w+?)_v(?P<wersja>\d+)_(?P<data>\d{4}_\d{2}_\d{2})\.txt")
for przyklad in ["humidity_ST02_widzew_v1_2025_03_05.txt", "humidity_ST01_polesie_v2_2025_03_07.txt"]:
    print(przyklad, "->", wzorzec_hum.match(przyklad).groupdict())

pm25-ST01_polesie-20250301.csv -> {'stacja': 'ST01_polesie', 'data': '20250301'}
pm25-ST04_gorna-20250308.csv -> {'stacja': 'ST04_gorna', 'data': '20250308'}
humidity_ST02_widzew_v1_2025_03_05.txt -> {'stacja': 'ST02_widzew', 'wersja': '1', 'data': '2025_03_05'}
humidity_ST01_polesie_v2_2025_03_07.txt -> {'stacja': 'ST01_polesie', 'wersja': '2', 'data': '2025_03_07'}


---

## 5. Łączymy: pathlib + regex + pandas

Teraz właściwa robota. Strategia:
1. Znajdź pliki danego typu (`rglob` + regex do walidacji nazwy)
2. Wyciągnij metadane z nazwy
3. Wczytaj do pandas
4. Dodaj kolumny z metadanymi
5. Złóż wszystko w jedną ramkę

In [13]:
import pandas as pd

def wczytaj_temperature(root: Path) -> pd.DataFrame:
    """Wczytuje wszystkie pliki temperatury, dodaje metadane, łączy w jedną ramkę."""
    ramki = []
    for f in root.rglob("temp_*.csv"):
        m = wzorzec_temp.match(f.name)
        if not m:
            print(f"⚠️  Pomijam: {f.name}")
            continue
        meta = m.groupdict()
        df = pd.read_csv(f)
        df["stacja"] = meta["stacja"]
        df["data_pliku"] = meta["data"]
        df["wersja"] = meta.get("wersja") or "1"
        df["plik"] = f.name
        ramki.append(df)
    return pd.concat(ramki, ignore_index=True)

temp_df = wczytaj_temperature(ROOT)
print("Kształt:", temp_df.shape)
temp_df.head()

Kształt: (320, 7)


,timestamp,temperatura_C,uwagi,stacja,data_pliku,wersja,plik
0,2025-03-01 00:00:00,-1.5,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
1,2025-03-01 03:00:00,1.69,deszcz,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
2,2025-03-01 06:00:00,17.64 C,ok,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
3,2025-03-01 09:00:00,1.34 C,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
4,2025-03-01 12:00:00,11.71 C,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv


In [14]:
print("Typ kolumny:", temp_df["temperatura_C"].dtype)
print("\nPodejrzane wartości:")
podejrzane = temp_df[temp_df["temperatura_C"].astype(str).str.contains("C", na=False)]
podejrzane.head()

Typ kolumny: object

Podejrzane wartości:


,timestamp,temperatura_C,uwagi,stacja,data_pliku,wersja,plik
2,2025-03-01 06:00:00,17.64 C,ok,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
3,2025-03-01 09:00:00,1.34 C,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
4,2025-03-01 12:00:00,11.71 C,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
14,2025-03-02 18:00:00,7.25 C,deszcz,ST01_polesie,2025-03-02,1,temp_ST01_polesie_2025-03-02.csv
47,2025-03-06 21:00:00,23.24 C,ok,ST01_polesie,2025-03-06,1,temp_ST01_polesie_2025-03-06.csv


## 6. `pandas.Series.str` — czyszczenie kolumn tekstowych

Akcesor `.str` w pandas to **odpowiednik metod stringa, ale zwektoryzowany**. Najważniejsze:

In [15]:
# Konwersja na string i usunięcie tekstu + spacji
temp_df["temperatura_C"] = (
    temp_df["temperatura_C"]
    .astype(str)
    .str.replace("C", "", regex=False)  # usuń literę C
    .str.strip()                          # usuń białe znaki z brzegów
    .astype(float)                        # teraz można rzutować
)

print("Typ po czyszczeniu:", temp_df["temperatura_C"].dtype)
print(temp_df["temperatura_C"].describe())

Typ po czyszczeniu: float64
count    320.000000
mean      10.938094
std        6.454670
min       -1.750000
25%        5.570000
50%       10.695000
75%       16.310000
max       24.250000
Name: temperatura_C, dtype: float64


### `str.extract` z grupami nazwanymi — najpotężniejsza metoda

Wyobraźcie sobie, że chcemy **rozbić nazwę stacji** `ST01_polesie` na kod numeryczny i nazwę dzielnicy:

In [16]:
rozbita = temp_df["stacja"].str.extract(r"ST(?P<kod>\d+)_(?P<dzielnica>\w+)")
rozbita.head()

,kod,dzielnica
0,01,polesie
1,01,polesie
2,01,polesie
3,01,polesie
4,01,polesie


In [17]:
# Dołączamy te kolumny do głównej ramki
temp_df = pd.concat([temp_df, rozbita], axis=1)
temp_df.head()

,timestamp,temperatura_C,uwagi,stacja,data_pliku,wersja,plik,kod,dzielnica
0,2025-03-01 00:00:00,-1.50,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv,01,polesie
1,2025-03-01 03:00:00,1.69,deszcz,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv,01,polesie
2,2025-03-01 06:00:00,17.64,ok,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv,01,polesie
3,2025-03-01 09:00:00,1.34,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv,01,polesie
4,2025-03-01 12:00:00,11.71,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv,01,polesie


### **ćwiczenie 4**
Napisz funkcję `wczytaj_pm25(root)` analogiczną do `wczytaj_temperature`.
Uwaga:
- separator w plikach to `;` (parametr `sep` w `read_csv`)
- data w nazwie jest w formacie `YYYYMMDD` — sparsuj ją do `pd.Timestamp`
- kolumna `czas` w środku ma format `DD.MM.YYYY HH:MM` — sparsuj do datetime

In [18]:
# Ćwiczenie 4 — wczytaj_pm25 (analogiczna do wczytaj_temperature)
def wczytaj_pm25(root: Path) -> pd.DataFrame:
    """Wczytuje pliki PM2.5, dodaje metadane z nazwy, łączy w jedną ramkę."""
    ramki = []
    for f in root.rglob("pm25-*.csv"):
        m = wzorzec_pm.match(f.name)          # wzorzec z ćwiczenia 3
        if not m:
            print("⚠️  Pomijam:", f.name)
            continue
        meta = m.groupdict()
        df = pd.read_csv(f, sep=";")                                   # separator ';'
        df["stacja"] = meta["stacja"]
        df["data_pliku"] = pd.to_datetime(meta["data"], format="%Y%m%d")  # YYYYMMDD z nazwy
        df["czas"] = pd.to_datetime(df["czas"], format="%d.%m.%Y %H:%M")  # DD.MM.YYYY HH:MM
        df["plik"] = f.name
        ramki.append(df)
    return pd.concat(ramki, ignore_index=True)

pm_df = wczytaj_pm25(ROOT)
print("Kształt:", pm_df.shape)
print(pm_df.dtypes)
pm_df.head()

Kształt: (160, 6)
czas           datetime64[us]
PM2.5_ug_m3           float64
PM10_ug_m3            float64
stacja                    str
data_pliku     datetime64[us]
plik                      str
dtype: object


,czas,PM2.5_ug_m3,PM10_ug_m3,stacja,data_pliku,plik
0,2025-03-01 00:00:00,23.8,44.7,ST01_polesie,2025-03-01,pm25-ST01_polesie-20250301.csv
1,2025-03-01 06:00:00,19.9,30.0,ST01_polesie,2025-03-01,pm25-ST01_polesie-20250301.csv
2,2025-03-01 12:00:00,41.2,57.3,ST01_polesie,2025-03-01,pm25-ST01_polesie-20250301.csv
3,2025-03-01 18:00:00,47.9,79.5,ST01_polesie,2025-03-01,pm25-ST01_polesie-20250301.csv
4,2025-03-02 00:00:00,79.5,114.2,ST01_polesie,2025-03-02,pm25-ST01_polesie-20250302.csv


---

## 7. `pd.concat` z `keys` — MultiIndex na łączeniu

Czasem chcemy zachować informację, **z którego pliku** pochodzi wiersz. Można dodać kolumnę (jak wyżej), albo użyć MultiIndex:

In [19]:
kawalki = {}
for f in list((ROOT / "temperatura").glob("temp_*.csv"))[:3]:
    kawalki[f.stem] = pd.read_csv(f)

duza = pd.concat(kawalki, names=["plik_zrodlowy", "wiersz"])
duza.head(7)

timestamp temperatura_C   uwagi
plik_zrodlowy                wiersz                                           
temp_ST01_polesie_2025-03-01 0       2025-03-01 00:00:00          -1.5     NaN
                             1       2025-03-01 03:00:00          1.69  deszcz
                             2       2025-03-01 06:00:00     17.64 C        ok
                             3       2025-03-01 09:00:00      1.34 C       NaN
                             4       2025-03-01 12:00:00     11.71 C       NaN
                             5       2025-03-01 15:00:00         16.82      ok
                             6       2025-03-01 18:00:00          7.81     NaN

---

## 8. Podsumowanie — pełny pipeline

Złożenie wszystkiego w jedną funkcję, która zwraca **gotową do analizy ramkę** ze wszystkich źródeł:

In [20]:
def zbuduj_dataset(root: Path) -> pd.DataFrame:
    """Łączy temperatury i PM2.5 w jedną ramkę po (stacja, czas)."""
    temp = wczytaj_temperature(root)
    # kolumna temperatura_C bywa brudna ("19.23 C") — czyścimy przed agregacją
    temp["temperatura_C"] = (temp["temperatura_C"].astype(str)
                             .str.replace("C", "", regex=False).str.strip().astype(float))
    temp["timestamp"] = pd.to_datetime(temp["timestamp"])
    # użyjemy zaokrąglenia do godziny - bo PM ma co 6h, temp co 3h
    temp["czas_h"] = temp["timestamp"].dt.floor("6h")

    pm = wczytaj_pm25(root) # funckja z poprzedniego ćwiczenia
    pm["czas_h"] = pm["czas"].dt.floor("6h")

    # Agregacja temperatury do okien 6h
    temp_agg = temp.groupby(["stacja", "czas_h"], as_index=False).agg(
        temp_srednia=("temperatura_C", "mean")
    )

    polaczone = pm.merge(temp_agg, on=["stacja", "czas_h"], how="left")
    return polaczone[["stacja", "czas_h", "PM2.5_ug_m3", "PM10_ug_m3", "temp_srednia"]]

final = zbuduj_dataset(ROOT)
print("Kształt finalnej ramki:", final.shape)
final.head(10)

Kształt finalnej ramki:

 (160, 5)


,stacja,czas_h,PM2.5_ug_m3,PM10_ug_m3,temp_srednia
0,ST01_polesie,2025-03-01 00:00:00,23.8,44.7,0.095
1,ST01_polesie,2025-03-01 06:00:00,19.9,30.0,9.490
2,ST01_polesie,2025-03-01 12:00:00,41.2,57.3,14.265
3,ST01_polesie,2025-03-01 18:00:00,47.9,79.5,13.645
4,ST01_polesie,2025-03-02 00:00:00,79.5,114.2,1.025
5,ST01_polesie,2025-03-02 06:00:00,78.3,135.2,16.020
6,ST01_polesie,2025-03-02 12:00:00,25.6,42.3,9.025
7,ST01_polesie,2025-03-02 18:00:00,56.4,101.3,8.055
8,ST01_polesie,2025-03-03 00:00:00,8.7,14.7,13.470
9,ST01_polesie,2025-03-03 06:00:00,42.3,81.4,13.170


## **Zadanie**

Dopisz funkcję `wczytaj_wilgotnosc(root)` dla plików w `wilgotnosc/`.
Uwagi:
1. Pliki mają **nagłówek z komentarzami** (linie zaczynające się od `#`) — pomiń je w `read_csv` parametrem `comment="#"`.
2. Data w nazwie jest w formacie `YYYY_MM_DD` (podkreślniki!) — przekonwertuj.
3. Niektóre pliki mają wersję `v2` — w finalnej ramce zostaw tylko **najnowszą wersję** dla każdej (stacja, data).
4. Dołącz wilgotność do `final` z poprzedniego kroku.

**Bonus:** wykryj duplikaty pliku temperatury (`_v2`) i zostaw tylko nowszą wersję — analogicznie.

In [21]:
# ✅ Zadanie — wczytaj_wilgotnosc
def wczytaj_wilgotnosc(root: Path) -> pd.DataFrame:
    """Wczytuje pliki wilgotności, parsuje metadane z nazwy, zostawia najnowszą wersję."""
    ramki = []
    for f in (root / "wilgotnosc").glob("humidity_*.txt"):
        m = wzorzec_hum.match(f.name)      # wzorzec z ćwiczenia 3
        if not m:
            print("⚠️  Pomijam:", f.name)
            continue
        meta = m.groupdict()
        # 1. pomijamy linie-komentarze zaczynające się od '#'
        df = pd.read_csv(f, comment="#")
        df["stacja"] = meta["stacja"]
        df["wersja"] = int(meta["wersja"])
        # 2. data w nazwie: YYYY_MM_DD (podkreślniki!) -> Timestamp
        data = pd.to_datetime(meta["data"], format="%Y_%m_%d")
        df["data_pliku"] = data
        # timestamp pomiaru = data + godzina (kolumna hour: 00, 04, 08, ...)
        df["czas"] = data + pd.to_timedelta(df["hour"].astype(int), unit="h")
        df["plik"] = f.name
        ramki.append(df)
    hum = pd.concat(ramki, ignore_index=True)
    # 3. zostaw tylko NAJNOWSZĄ wersję dla każdej (stacja, data/godzina)
    hum = (hum.sort_values("wersja")
              .drop_duplicates(subset=["stacja", "czas"], keep="last")
              .reset_index(drop=True))
    return hum

wilg = wczytaj_wilgotnosc(ROOT)
print("Kształt wilgotności:", wilg.shape)
print("Wersje w danych:", sorted(wilg["wersja"].unique()))
wilg.head()

Kształt wilgotności: (240, 7)
Wersje w danych: [np.int64(1), np.int64(2)]


,hour,humidity_pct,stacja,wersja,data_pliku,czas,plik
0,0,59.3,ST01_polesie,1,2025-03-01,2025-03-01 00:00:00,humidity_ST01_polesie_v1_2025_03_01.txt
1,4,92.7,ST01_polesie,1,2025-03-01,2025-03-01 04:00:00,humidity_ST01_polesie_v1_2025_03_01.txt
2,8,75.5,ST01_polesie,1,2025-03-01,2025-03-01 08:00:00,humidity_ST01_polesie_v1_2025_03_01.txt
3,12,48.1,ST01_polesie,1,2025-03-01,2025-03-01 12:00:00,humidity_ST01_polesie_v1_2025_03_01.txt
4,16,45.7,ST01_polesie,1,2025-03-01,2025-03-01 16:00:00,humidity_ST01_polesie_v1_2025_03_01.txt


In [22]:
# 4. Dołączamy wilgotność do finalnej ramki `final`
# Agregujemy do okien 6h (jak temperaturę), bo wilgotność jest co 4h
wilg["czas_h"] = wilg["czas"].dt.floor("6h")
wilg_agg = wilg.groupby(["stacja", "czas_h"], as_index=False).agg(
    wilgotnosc_srednia=("humidity_pct", "mean")
)

final2 = final.merge(wilg_agg, on=["stacja", "czas_h"], how="left")
print("Finalna ramka z wilgotnością:", final2.shape)
print("Braki wilgotności:", int(final2["wilgotnosc_srednia"].isna().sum()), "/", len(final2))
final2.head(10)

Finalna ramka z wilgotnością: (160, 6)
Braki wilgotności: 0 / 160


,stacja,czas_h,PM2.5_ug_m3,PM10_ug_m3,temp_srednia,wilgotnosc_srednia
0,ST01_polesie,2025-03-01 00:00:00,23.8,44.7,0.095,76.00
1,ST01_polesie,2025-03-01 06:00:00,19.9,30.0,9.490,75.50
2,ST01_polesie,2025-03-01 12:00:00,41.2,57.3,14.265,46.90
3,ST01_polesie,2025-03-01 18:00:00,47.9,79.5,13.645,54.10
4,ST01_polesie,2025-03-02 00:00:00,79.5,114.2,1.025,77.45
5,ST01_polesie,2025-03-02 06:00:00,78.3,135.2,16.020,76.10
6,ST01_polesie,2025-03-02 12:00:00,25.6,42.3,9.025,66.10
7,ST01_polesie,2025-03-02 18:00:00,56.4,101.3,8.055,67.20
8,ST01_polesie,2025-03-03 00:00:00,8.7,14.7,13.470,83.00
9,ST01_polesie,2025-03-03 06:00:00,42.3,81.4,13.170,75.40
